# Catalyst — frozen eval (offline)

Reads committed artifacts only: comparison JSON, one MCJ trace, frozen SQLite fingerprint. No network or provider APIs.

In [1]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from pathlib import Path

REPO = Path.cwd()
if (REPO / "data" / "eval_reports").is_dir():
    root = REPO
elif (REPO.parent / "data" / "eval_reports").is_dir():
    root = REPO.parent
    import os

    os.chdir(root)
else:
    raise RuntimeError("Run from repo root or under notebooks/ with parent containing data/eval_reports")

report_dir = root / "data" / "eval_reports"
comparisons = sorted(report_dir.glob("*_comparison.json"))
assert comparisons, "missing *_comparison.json"
comparison_path = comparisons[-1]
comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
header = comparison["header"]
print("comparison:", comparison_path.name)
print("frozen_ts:", header["frozen_ts"])
print("code_git_sha:", header["code_git_sha"][:12], "…")
print("gates:")
for k, v in comparison["gates"].items():
    print(f"  {k}: {v}")

comparison: 20260503_154053_comparison.json
frozen_ts: 20260503_154053
code_git_sha: 9f844eddffff …
gates:
  cost_latency_reported: True
  evidence_validity: 1.0
  schema_validity: 1.0
  should_refuse_hit_rate: 1.0
  trace_completeness: 1.0


In [2]:
db_path = root / "data" / "catalyst_eval_frozen.db"
digest = hashlib.sha256(db_path.read_bytes()).hexdigest()
assert digest == header["db_sha256"], "DB fingerprint drift vs comparison header"
print("db_sha256 OK (matches header)")

cl = comparison["cost_latency"]
print("avg_cost_usd direct_llm:", cl["direct_llm"]["avg_cost_usd"])
print("avg_cost_usd mcj_full:", cl["mcj_full"]["avg_cost_usd"])
print("avg_latency_ms direct_llm:", cl["direct_llm"]["avg_latency_ms"])
print("avg_latency_ms mcj_full:", cl["mcj_full"]["avg_latency_ms"])

db_sha256 OK (matches header)
avg_cost_usd direct_llm: 0.0011268000000000003
avg_cost_usd mcj_full: 0.0026544000000000003
avg_latency_ms direct_llm: 4.3
avg_latency_ms mcj_full: 4.7


In [3]:
first = comparison["per_case"][0]
run_id = first["mcj_full"]["run_id"]
trace_path = root / "data" / "traces" / f"{run_id}.json"
trace = json.loads(trace_path.read_text(encoding="utf-8"))
print("sample case_id:", first["case_id"])
print("trace_path:", trace_path.relative_to(root))
print("trace_id:", trace["trace_id"])
print("events:", len(trace.get("events", [])))

sample case_id: g006
trace_path: data/traces/019acafe4dc744e8b9628b51f8b13120.json
trace_id: cbf401da13eb452d9b7cad1aeec21dc3
events: 6


In [4]:
venv_python = root / "packages" / "data-core" / ".venv" / "bin" / "python"
gate_script = root / "packages" / "eval" / "scripts" / "check_p0_gate.py"
if not venv_python.is_file():
    venv_python = Path(sys.executable)
proc = subprocess.run(
    [str(venv_python), str(gate_script), str(comparison_path)],
    cwd=str(root),
    capture_output=True,
    text=True,
)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr, file=sys.stderr)
print("check_p0_gate exit:", proc.returncode)
assert proc.returncode == 0, "Gate script did not return GREEN (0)"

comparison_report=/Users/yiannischen/Desktop/Catalyst/data/eval_reports/20260503_154053_comparison.json
| Gate | Actual | Target | Result |
|---|---:|---:|---|
| evidence_validity | 1.0 | >= 0.95 | PASS |
| schema_validity | 1.0 | == 1.0 | PASS |
| trace_completeness | 1.0 | == 1.0 | PASS |
| should_refuse_hit_rate | 1.0 | >= 0.6666666666666666 | PASS |
| cost_latency_reported | True | == True | PASS |
OVERALL=GREEN

check_p0_gate exit: 0
